# Notebook 2: Pollution Prediction with Machine Learning
## AI in Environmental Chemistry

Trains XGBoost and Random Forest models to predict Air Quality Index (AQI).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_sample_air_quality
from src.data.preprocessor import (
    clean_air_quality, engineer_air_features,
    encode_categoricals, split_features_target
)
from src.models.pollution_predictor import PollutionPredictor
from src.visualization.plots import plot_feature_importance, plot_actual_vs_predicted

print('Ready.')

In [ ]:
# Prepare data
df = load_sample_air_quality()
df = clean_air_quality(df)
df = engineer_air_features(df)
df = encode_categoricals(df, columns=['location', 'season', 'pollution_level'])

FEATURE_COLS = ['PM2_5', 'PM10', 'NO2', 'O3', 'CO',
                'temperature_C', 'humidity_pct', 'wind_speed_ms',
                'month', 'day_of_week', 'heat_stagnation']

X, y = split_features_target(df, target='AQI', drop_cols=['date'])
X = X[[c for c in FEATURE_COLS if c in X.columns]]

print('Feature matrix:', X.shape)
print('Target range:', round(y.min(), 1), '—', round(y.max(), 1))

In [ ]:
# Train XGBoost predictor
xgb_model = PollutionPredictor(model_type='xgboost')
metrics_xgb = xgb_model.train(X, y)
print('XGBoost metrics:', metrics_xgb)

In [ ]:
# Train Random Forest predictor
rf_model = PollutionPredictor(model_type='random_forest')
metrics_rf = rf_model.train(X, y)
print('Random Forest metrics:', metrics_rf)

In [ ]:
# Cross-validation comparison
cv_xgb = xgb_model.cross_validate(X, y)
cv_rf  = rf_model.cross_validate(X, y)

print('XGBoost CV:', cv_xgb)
print('Random Forest CV:', cv_rf)

In [ ]:
# Feature importances — XGBoost
importances = xgb_model.feature_importances(list(X.columns))
fig = plot_feature_importance(importances, title='XGBoost — AQI Feature Importances')
plt.show()

In [ ]:
# Actual vs predicted
y_pred = xgb_model.predict(X)
fig = plot_actual_vs_predicted(y.values, y_pred, label='AQI')
plt.show()

In [ ]:
# Predict AQI for a new sample
import pandas as pd
new_sample = pd.DataFrame([{
    'PM2_5': 85.0, 'PM10': 120.0, 'NO2': 60.0, 'O3': 80.0, 'CO': 1.2,
    'temperature_C': 28.0, 'humidity_pct': 70.0, 'wind_speed_ms': 1.5,
    'month': 7, 'day_of_week': 2, 'heat_stagnation': 28.0 / (1.5 + 0.1)
}])

predicted_aqi = xgb_model.predict(new_sample)[0]
print(f'Predicted AQI: {predicted_aqi:.1f}')
if predicted_aqi < 50:
    print('Level: Good')
elif predicted_aqi < 100:
    print('Level: Moderate')
else:
    print('Level: Unhealthy')